In [18]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [19]:
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [36]:
# all the imports
import os
import pandas as pd
import numpy as np

from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder as ImageFolder
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

Afmetingen aangepast Trainingset

In [21]:
class GefilterdeTumorDataset(Dataset):
    def __init__(self, root, min=256, transform=None): # min is minimale grens en root is hoofdmap
        # Haal de submappen op als klassen
        self.classes = [
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ]

        self.images = []
        self.transform = transform

        for index, group in enumerate(self.classes):
            group_map = os.path.join(root, group)

            for filename in os.listdir(group_map):
                if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    pad = os.path.join(group_map, filename)

                    # Controleer de afmetingen
                    with Image.open(pad) as img:
                        width, height = img.size

                    if width >= min and height >= min:
                        self.images.append((pad, index))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        pad, label = self.images[idx]
        img = Image.open(pad).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label


In [22]:
# Toepassing in jouw bestaande pipeline
transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

In [44]:
dataset_train = GefilterdeTumorDataset(root='./Training', min=256, transform=transform)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_train)}")

Totaal aantal gefilterde foto's ingeladen: 2559


In [45]:
dataset_test = GefilterdeTumorDataset(root='./Testing', min=256, transform=transform)
dataloader_test = DataLoader(dataset_test, batch_size=32, shuffle=True)

print(f"Totaal aantal gefilterde foto's ingeladen: {len(dataset_test)}")

Totaal aantal gefilterde foto's ingeladen: 212


In [25]:
%pip install numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install torch torchvision pandas pillow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [57]:
# Bestandslocaties
csv_path = "./project-1-at-2026-10-28-09-28-c53f7012.csv"

# CSV inlezen
df = pd.read_csv(csv_path)

print("CSV kolommen:", df.columns.tolist())
print(df.head())

# Labels koppelen aan de klassen van je Dataset
label_mapping = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}

# Bestandsnamen uit de bestaande Dataset
image_paths = [item[0] for item in dataset_train.images]

filenames = [
    os.path.basename(path)
    for path in image_paths
]

# CSV-labels normaliseren
df["label"] = df["choice"].astype(str).str.strip().str.lower()

# Labels uit CSV ophalen
label_dict = dict(zip(df["image"], df["choice"]))

# Labels maken
y = []

for filename in filenames:

    if filename in label_dict:
        label = label_mapping[label_dict[filename]]
    else:
        label = -1

    y.append(label)

y = np.array(y)

print("\nAantal afbeeldingen:", len(filenames))
print("y shape:", y.shape)
print("Labels:", np.unique(y, return_counts=True))

print("Aantal gelabeld:", np.sum(y != -1))
print("Aantal ongelabeld:", np.sum(y == -1))

CSV kolommen: ['annotation_id', 'annotator', 'choice', 'created_at', 'id', 'image', 'lead_time', 'updated_at']
   annotation_id  annotator     choice                   created_at  id  \
0             19          1     Axiaal  2026-10-27T13:08:58.955923Z   1   
1             34          1     Axiaal  2026-10-27T13:10:02.916840Z   2   
2              2          1  Sagittaal  2026-10-27T13:07:19.266567Z   3   
3             26          1  Sagittaal  2026-10-27T13:09:31.761523Z   4   
4             24          1  Sagittaal  2026-10-27T13:09:17.653864Z   5   

                               image  lead_time                   updated_at  
0  /data/upload/1/abb28261-gg_18.jpg      3.591  2026-10-27T13:08:58.955949Z  
1  /data/upload/1/1249a5d9-gg_22.jpg      3.197  2026-10-27T13:10:02.916864Z  
2  /data/upload/1/352806dd-gg_32.jpg      3.645  2026-10-27T13:07:19.266592Z  
3  /data/upload/1/436e3a05-gg_31.jpg      9.018  2026-10-27T13:09:31.761543Z  
4  /data/upload/1/40291c3e-gg_48.jpg      3

In [62]:
print("CSV afbeeldingen:")
print(df["image"].head(10).tolist())

print("\nDataset afbeeldingen:")
print(dataset_train.images[:10])

print("\nCSV labels:")
print(df["label"].unique())

CSV afbeeldingen:
['/data/upload/1/abb28261-gg_18.jpg', '/data/upload/1/1249a5d9-gg_22.jpg', '/data/upload/1/352806dd-gg_32.jpg', '/data/upload/1/436e3a05-gg_31.jpg', '/data/upload/1/40291c3e-gg_48.jpg', '/data/upload/1/4a80d6a2-gg_155.jpg', '/data/upload/1/a607946c-gg_199.jpg', '/data/upload/1/c6fc8699-gg_329.jpg', '/data/upload/1/4c4a0a3e-gg_387.jpg', '/data/upload/1/3e50bdff-gg_402.jpg']

Dataset afbeeldingen:
[('./Training/no_tumor/image(251).jpg', 0), ('./Training/no_tumor/image (26).jpg', 0), ('./Training/no_tumor/image(210).jpg', 0), ('./Training/no_tumor/image (30).jpg', 0), ('./Training/no_tumor/image(181).jpg', 0), ('./Training/no_tumor/image(247).jpg', 0), ('./Training/no_tumor/image(302).jpg', 0), ('./Training/no_tumor/image(230).jpg', 0), ('./Training/no_tumor/image(288).jpg', 0), ('./Training/no_tumor/image(322).jpg', 0)]

CSV labels:
['axiaal' 'sagittaal' 'coronaal' 'nan']


In [63]:
base_classifier = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    max_iter=10000,
    class_weight='balanced',
    l1_ratio=0.1,
    C=0.001
)

self_training_clf = SelfTrainingClassifier(
    base_classifier,
    criterion="k_best",
    k_best=2,
    max_iter=10
)

self_training_clf.fit(X, y)

ValueError: Found input variables with inconsistent numbers of samples: [2870, 2559]